In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import matplotlib
import matplotlib
# Step 1: Load the main CSV file
file_path = '<EXTERNAL_DATA_ROOT>'
data = pd.read_csv(file_path)

# Step 2: Clean the data (remove 'unknown' columns if present)
data_cleaned = data.loc[:, ~data.columns.str.contains("unknown", case=False)]

# Step 3: Set 'Model' column as index
data_cleaned.set_index('Model', inplace=True)

# Step 4: Calculate sums for models and compounds
model_sums = data_cleaned.sum(axis=1)
compound_sums = data_cleaned.sum(axis=0)

# Step 5: Sort models and compounds
data_sorted = data_cleaned.loc[model_sums.sort_values(ascending=False).index, 
                               compound_sums.sort_values(ascending=False).index]

# Step 6: Transpose the data for the heatmap
data_transposed = data_sorted.T

# Step 7: Load the Excel file and parse the "BBBNames" sheet
excel_file_path = '<EXTERNAL_DATA_ROOT>'
excel_data = pd.ExcelFile(excel_file_path)
bbb_names_data = excel_data.parse('BBBNames')

# Step 8: Create a mapping of IDs to names
id_name_mapping = dict(zip(bbb_names_data['ID'], bbb_names_data['Name']))

# Step 9: Rename compound IDs to names in the transposed data
data_transposed_renamed = data_transposed.rename(index=id_name_mapping)

# Step 10: Plot the heatmap with mapped names
plt.figure(figsize=(16, 10))
sns.heatmap(data_transposed_renamed, cmap='Greys', cbar=True, linewidths=0.5)

# Title and labels
plt.title('Presence-Absence Heatmap (With Mapped Names)', fontsize=14)
plt.xlabel('Models', fontsize=12)
plt.ylabel('Compounds (Mapped)', fontsize=12)

# Show the heatmap
plt.tight_layout()
plt.savefig('Heatmap_BBBs.pdf', format='pdf', bbox_inches='tight')
plt.show()
import numpy as np

# Step 1: Load the Excel file and parse Sheet 1
excel_file_path = '<EXTERNAL_DATA_ROOT>'
excel_data = pd.ExcelFile(excel_file_path)
sheet1_data = excel_data.parse('Number')  # Assuming 'Number' is Sheet 1

# Step 2: Calculate "Relaxed_Difference" as the difference between "Constrained" and "Relaxed"
sheet1_data['Relaxed_Difference'] = sheet1_data['Constrained'] - sheet1_data['Relaxed']

# Step 3: Sort models based on "Constrained" values for better visualization
sheet1_data_sorted = sheet1_data.sort_values(by='Constrained', ascending=False)

# Step 4: Plot a stacked bar chart
plt.figure(figsize=(12, 8))
plt.bar(sheet1_data_sorted['Model'], sheet1_data_sorted['Relaxed'], label='Relaxed', color='skyblue')
plt.bar(sheet1_data_sorted['Model'], sheet1_data_sorted['Relaxed_Difference'], 
        bottom=sheet1_data_sorted['Relaxed'], label='Remaining (Constrained)', color='darkblue')

# Step 5: Titles and labels
plt.title('Stacked Bar Plot of Relaxed and Constrained Conditions', fontsize=14)
plt.xlabel('Models', fontsize=12)
plt.ylabel('Counts', fontsize=12)
plt.xticks(rotation=45, ha='right')
plt.legend(title='Condition')

# Step 6: Ensure y-axis scale shows only integers
plt.yticks(np.arange(0, sheet1_data_sorted['Constrained'].max() + 1, 1))

# Step 7: Show the plot
plt.tight_layout()
plt.savefig('Stacked_Bar_Plot.pdf', format='pdf', bbox_inches='tight')
plt.show()


In [ ]:
#!/usr/bin/env python3
"""
stacked_gapfill_media.py  ·  clean data · Schwabish-style stacked bars
----------------------------------------------------------------------
• Forward-fills blank Media cells (inherit previous tag)
• Skips spacer rows (no Rxns ID)
• Draws colour-blind-safe stacked bars, sorted by total reactions
"""

import re
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib
# ─── 1. Paths ─────────────────────────────────────────────────────────
ODS_PATH = Path("./GFUnified.ods")          # adjust if needed
PDF_PATH = Path("./gapfillRxns.pdf")

# ─── 2. Minimal-ink, journal style (Schwabish ch. 7) ─────────────────
plt.rcParams.update({
    "font.family": "serif",
    "font.size":   10,
    "axes.labelsize": 10,
    "axes.titlesize": 12,
    "legend.fontsize": 8,
    "figure.dpi": 300,
})

# ─── 3. Read workbook and detect organism sheets ─────────────────────
xls = pd.ExcelFile(ODS_PATH, engine="odf")
organism_sheets = [s for s in xls.sheet_names if s.lower() != "scoring"]

media_counts = {}      # {organism_code: {medium: count}}

for sheet in organism_sheets:
    df = xls.parse(sheet)

    # 3a. Find Media column (cuvk fix)
    if "Media" not in df.columns and len(df.columns) > 10:
        df = df.rename(columns={df.columns[10]: "Media"})

    # 3b. Forward-fill Media, strip whitespace
    df["Media"] = df["Media"].ffill().astype(str).str.strip()

    # 3c. Drop spacer rows (blank Rxns)
    df = df[df["Rxns"].notna()]

    counts = {}
    for val in df["Media"]:
        # Split by common delimiters; skip empty tokens
        for token in re.split(r"[,\;/|]+", val):
            token = token.strip()
            if token:
                counts[token] = counts.get(token, 0) + 1

    media_counts[sheet] = counts

# ─── 4. Build tidy DataFrame for plotting ────────────────────────────
all_media = sorted({m for d in media_counts.values() for m in d})
df = pd.DataFrame(index=sorted(media_counts.keys()), columns=all_media).fillna(0)
for org, counts in media_counts.items():
    for med, n in counts.items():
        df.loc[org, med] = n

# ─── 5. Sort organisms by total reactions (easier comparison) ────────
df = df.loc[df.sum(axis=1).sort_values(ascending=False).index]

# ─── 6. Draw stacked bar chart (colour-blind safe) ───────────────────
fig, ax = plt.subplots(figsize=(7, 3.2))
bottom = pd.Series(0, index=df.index)

for med in all_media:
    ax.bar(df.index, df[med], bottom=bottom, label=med)
    bottom += df[med]

# Labelling & styling in line with the book’s guidance
ax.set_ylabel("Gap-fill reactions")
ax.set_xlabel("Organism code")
ax.set_title("Gap-fill reaction distribution across growth media")
ax.set_xticklabels(df.index, rotation=45, ha="right")
ax.grid(axis="y", linestyle=":", linewidth=0.5, zorder=0)
ax.legend(ncol=1, bbox_to_anchor=(1.02, 1), loc="upper left", frameon=False)

fig.tight_layout()
fig.savefig(PDF_PATH, format="pdf", bbox_inches="tight")
print("Saved figure to:", PDF_PATH)


In [ ]:
#!/usr/bin/env python3
"""
stacked_gapfill_media_grouped.py

• Reads GFUnified.ods per-organism sheets
• Counts gap-fill reactions per medium
• Groups organisms by phylum and, within each phylum,
  orders them by total gap-fill reactions (high → low)
• Stacks media with:
    - PTYG at the bottom
    - Succinate above PTYG
    - Remaining media in order of first appearance
• NO phylum labels drawn (only organism codes on x-axis)
"""

import re
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import matplotlib
matplotlib.use("cairo")  # for headless environments
# ─── 0. Phylum mapping (edit to your exact mapping) ───────────────────
# === Define phyla mapping ===
phyla_dict = {
    "cohm": "Firmicutes",
    "cuvk": "Actinobacteria",
    "cxy": "Actinobacteria",
    "mip": "Actinobacteria",
    "rfa": "Actinobacteria",
    "cpi": "Bacteroidetes",
    "flk": "Bacteroidetes",
    "mgk": "Bacteroidetes",
    "bbet": "Alphaproteobacteria",
    "buol": "Betaproteobacteria",
    "cauj": "Alphaproteobacteria",
    "dri": "Alphaproteobacteria",
    "lgu": "Gammaproteobacteria",
    "lur": "Gammaproteobacteria",
    "mlot": "Alphaproteobacteria",
    "pmor": "Gammaproteobacteria",
    "pscf": "Gammaproteobacteria",
    "pzu": "Alphaproteobacteria",
    "racd": "Gammaproteobacteria",
    "tro": "Alphaproteobacteria",
    "vpe": "Betaproteobacteria",
}


# Order of phyla on the x-axis
phyla_order = [
    "Actinobacteria",
    "Alphaproteobacteria",
    "Betaproteobacteria",
    "Gammaproteobacteria",
    "Firmicutes",
    "Bacteroidetes"
]

# ─── 1. Paths ─────────────────────────────────────────────────────────
ODS_PATH = Path("./GFUnified.ods")          # adjust if needed
PDF_PATH = Path("./gapfillRxns_grouped.pdf")

# ─── 2. Minimal-ink, journal style ───────────────────────────────────
plt.rcParams.update({
    "font.family": "serif",
    "font.size":   10,
    "axes.labelsize": 10,
    "axes.titlesize": 12,
    "legend.fontsize": 8,
    "figure.dpi": 300,
})

# ─── 3. Read workbook and detect organism sheets ─────────────────────
xls = pd.ExcelFile(ODS_PATH, engine="odf")
organism_sheets = [s for s in xls.sheet_names if s.lower() != "scoring"]

media_counts = {}      # {organism_code: {medium: count}}

for sheet in organism_sheets:
    df_sheet = xls.parse(sheet)

    # 3a. Find Media column (cuvk fix)
    if "Media" not in df_sheet.columns and len(df_sheet.columns) > 10:
        df_sheet = df_sheet.rename(columns={df_sheet.columns[10]: "Media"})

    # 3b. Forward-fill Media, strip whitespace
    df_sheet["Media"] = df_sheet["Media"].ffill().astype(str).str.strip()

    # 3c. Drop spacer rows (blank Rxns)
    df_sheet = df_sheet[df_sheet["Rxns"].notna()]

    counts = {}
    for val in df_sheet["Media"]:
        # Split by common delimiters; skip empty tokens
        for token in re.split(r"[,\;/|]+", val):
            token = token.strip()
            if token:
                counts[token] = counts.get(token, 0) + 1

    media_counts[sheet] = counts

# ─── 4. Build tidy DataFrame for plotting ────────────────────────────
all_media = sorted({m for d in media_counts.values() for m in d})
df = pd.DataFrame(index=sorted(media_counts.keys()), columns=all_media).fillna(0)
for org, counts in media_counts.items():
    for med, n in counts.items():
        df.loc[org, med] = n

# ─── 5. Determine media stacking order ───────────────────────────────
# PTYG at bottom, Succinate above; others by first appearance.

preferred_first = ["PTYG", "Succinate"]

org_list = df.index.tolist()
appearance_order = {}

for med in all_media:
    if med in preferred_first:
        continue
    nonzero_orgs = [org for org in org_list if df.loc[org, med] > 0]
    if nonzero_orgs:
        first_idx = org_list.index(nonzero_orgs[0])
    else:
        first_idx = 10**9  # never appears
    appearance_order[med] = first_idx

other_media_sorted = sorted(appearance_order, key=appearance_order.get)

media_order = [m for m in preferred_first if m in all_media] + \
              [m for m in other_media_sorted if m in all_media]

# ─── 6. Group organisms by phylum and sort within phylum ─────────────
df["Phylum"] = df.index.to_series().map(phyla_dict).fillna("Unknown")
df["Total"] = df[all_media].sum(axis=1)

ordered_orgs = []
for phy in phyla_order:
    group = df[df["Phylum"] == phy].copy()
    if group.empty:
        continue
    group = group.sort_values("Total", ascending=False)
    ordered_orgs.extend(group.index.tolist())

# any organisms with phylum not in phyla_order (or Unknown) go at the end
rest = [org for org in df.index if org not in ordered_orgs]
ordered_orgs.extend(rest)

# apply ordering and drop helper columns from bars
df = df.loc[ordered_orgs].copy()
df_bars = df.drop(columns=["Phylum", "Total"])

# positions for bars
x = np.arange(len(df_bars.index))

# ─── 7. Draw stacked bar chart (no phylum labels) ────────────────────
fig, ax = plt.subplots(figsize=(7, 3.2))
bottom = pd.Series(0, index=df_bars.index)

for med in media_order:
    ax.bar(x, df_bars[med], bottom=bottom, label=med)
    bottom += df_bars[med]

# Labelling & styling
ax.set_ylabel("Gap-fill reactions")
ax.set_xlabel("Organism code")
ax.set_title("Gap-fill reaction distribution across growth media")

ax.set_xticks(x)
ax.set_xticklabels(df_bars.index, rotation=45, ha="right")

ax.grid(axis="y", linestyle=":", linewidth=0.5, zorder=0)
ax.legend(ncol=1, bbox_to_anchor=(1.02, 1), loc="upper left", frameon=False)

fig.tight_layout()
fig.savefig(PDF_PATH, format="pdf", bbox_inches="tight")
print("Saved figure to:", PDF_PATH)
